# 加密货币回测

拿币安公开行情，跑一个最简单的双均线策略回测，只看结果，不真交易。

不用选显卡，默认就行。点「运行时」→「全部运行」，等出结果。

In [ ]:
!pip install -q pandas numpy matplotlib requests

In [ ]:
import requests, pandas as pd, numpy as np

# 币安公开行情，不用 API Key
url = "https://api.binance.com/api/v3/klines"
params = {"symbol": "BTCUSDT", "interval": "1d", "limit": 1000}
data = requests.get(url, params=params, timeout=30).json()

df = pd.DataFrame(data, columns=[
    "time", "open", "high", "low", "close", "volume",
    "close_time", "quote_vol", "trades", "taker_buy_base",
    "taker_buy_quote", "ignore"])
df["date"] = pd.to_datetime(df["time"], unit="ms")
df["close"] = df["close"].astype(float)
df = df[["date", "close"]].set_index("date")
print("行情天数:", len(df))
print(df.tail(3))

In [ ]:
# 双均线策略：20 日均线在 60 日均线之上就持有，否则空仓
df["ma20"] = df["close"].rolling(20).mean()
df["ma60"] = df["close"].rolling(60).mean()
df["signal"] = (df["ma20"] > df["ma60"]).astype(int)
df["ret"] = df["close"].pct_change()
df["strat_ret"] = df["signal"].shift(1) * df["ret"]
df = df.dropna()

total_btc = (1 + df["ret"]).prod() - 1
total_strat = (1 + df["strat_ret"]).prod() - 1
equity = (1 + df["strat_ret"]).cumprod()
max_dd = (equity / equity.cummax() - 1).min()
win_rate = (df["strat_ret"][df["signal"].shift(1) == 1] > 0).mean()

print(f"时间：{df.index[0].date()} 到 {df.index[-1].date()}")
print(f"一直拿着 BTC：{total_btc:.1%}")
print(f"双均线策略：{total_strat:.1%}")
print(f"最大回撤：{max_dd:.1%}")
print(f"持仓天胜率：{win_rate:.1%}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax[0].plot(df.index, df["close"], label="BTC 价格")
ax[0].plot(df.index, df["ma20"], label="20 日均线")
ax[0].plot(df.index, df["ma60"], label="60 日均线")
ax[0].legend()
ax[0].set_title("BTC 价格与均线")
ax[1].plot(df.index, (1 + df["strat_ret"]).cumprod(), label="策略净值")
ax[1].plot(df.index, (1 + df["ret"]).cumprod(), label="一直持有")
ax[1].legend()
ax[1].set_title("策略净值 vs 一直持有")
plt.tight_layout()
plt.show()

这只是演示，回测不代表未来，不构成投资建议，也没有真交易。

想换币种，把 BTCUSDT 改成 ETHUSDT 之类的就行；想换均线天数，改 ma20、ma60 的数字。